In [2]:
from __future__ import annotations

from pathlib import Path
import shutil
import pandas as pd

# --- Input roots
ROOT_8000 = Path("data/8000")
ROOT_213 = Path("data/camera213_validated_images")
CSV_8000 = ROOT_8000 / "8000.csv"
CSV_213 = ROOT_213 / "camera213_validated_images.csv"

# --- Output root
OUT_ROOT = Path("data/img_all")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def read_csv_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Drop unnamed index column if exists
    if df.columns[0].startswith("Unnamed") or df.columns[0] == "":
        df = df.iloc[:, 1:]
    return df

def normalize_rel(p: str) -> str:
    # normalize to relative path under dataset root
    p = str(p).replace("\\", "/").lstrip("/")
    return p

def copy_path(src_root: Path, rel_path: str, dst_root: Path) -> bool:
    src = src_root / rel_path
    dst = dst_root / rel_path
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True

df_8000 = read_csv_clean(CSV_8000)
df_213 = read_csv_clean(CSV_213)

# Track counts
missing = 0
copied = 0

# Normalize and copy images for 8000
for col in ["image_name", "image_name_gray"]:
    if col in df_8000.columns:
        df_8000[col] = df_8000[col].astype(str).map(normalize_rel)
        for rel in df_8000[col].unique():
            if not copy_path(ROOT_8000, rel, OUT_ROOT):
                missing += 1
            else:
                copied += 1

# Normalize and copy images for camera213_validated_images
for col in ["image_name", "image_name_gray"]:
    if col in df_213.columns:
        df_213[col] = df_213[col].astype(str).map(normalize_rel)
        for rel in df_213[col].unique():
            if not copy_path(ROOT_213, rel, OUT_ROOT):
                missing += 1
            else:
                copied += 1

# Optional: add source column for traceability
df_8000["source_dir"] = "8000"
df_213["source_dir"] = "camera213_validated_images"

# Align columns + concat
all_cols = sorted(set(df_8000.columns) | set(df_213.columns))
df_8000 = df_8000.reindex(columns=all_cols)
df_213 = df_213.reindex(columns=all_cols)
df_all = pd.concat([df_8000, df_213], ignore_index=True)

# Save combined labels
out_csv = OUT_ROOT / "labels.csv"
df_all.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("Saved:", out_csv.resolve())
print("Total rows:", len(df_all))
print("Copied files:", copied)
print("Missing files:", missing)

Saved: D:\CodingD\ALPR\data_preprocessor\data\img_all\labels.csv
Total rows: 10320
Copied files: 20615
Missing files: 25
